# 05 - Evaluate Africa Level 2 Models and Make Plots

In [ ]:
from pathlib import Path
import sys
# Resolve local package imports from common notebook launch locations.
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT / 'src'))

import pandas as pd

from grace_gnn.config import REGION_FIGURES, REGION_IMPROVEMENT_BY_BASIN_CSV, REGION_METRICS_BY_BASIN_CSV, REGION_METRICS_OVERALL_CSV, REGION_OUTPUTS, REGION_PREDICTION_DIAGNOSTICS_CSV, REGION_PREDICTIONS_CSV, ensure_dirs
from grace_gnn.metrics import improvement_by_basin, metrics_by_basin, metrics_overall, prediction_diagnostics
from grace_gnn.plots import plot_graph_edges, plot_improvement_by_basin, plot_observed_vs_predicted, plot_rmse_bar, plot_time_series

ensure_dirs()
if not REGION_PREDICTIONS_CSV.exists():
# Load the combined prediction table produced by the training notebooks.
    print(f'Missing {REGION_PREDICTIONS_CSV}. Run notebooks 03 and 04 first.')
    predictions = None
else:
    predictions = pd.read_csv(REGION_PREDICTIONS_CSV, parse_dates=['date'])
    print(f'Loaded {len(predictions):,} Africa Level 2 predictions')

In [ ]:
if predictions is not None:
    # Compute aggregate, basin-level, and diagnostic evaluation tables.
    overall = metrics_overall(predictions)
    by_basin = metrics_by_basin(predictions, split='test')
    improvement = improvement_by_basin(by_basin)
    diagnostics = prediction_diagnostics(predictions)
    # Save metrics artifacts used for reporting and follow-up analysis.
    overall.to_csv(REGION_METRICS_OVERALL_CSV, index=False)
    by_basin.to_csv(REGION_METRICS_BY_BASIN_CSV, index=False)
    improvement.to_csv(REGION_IMPROVEMENT_BY_BASIN_CSV, index=False)
    diagnostics.to_csv(REGION_PREDICTION_DIAGNOSTICS_CSV, index=False)
    print('Saved metrics tables.')
    display(overall.sort_values(['split', 'rmse_cm']))
    display(improvement.head())

In [ ]:
if predictions is not None:
    # Generate the core evaluation figures from the saved prediction table.
    plot_time_series(predictions, REGION_FIGURES / 'timeseries_observed_vs_predicted.png')
    plot_rmse_bar(overall, REGION_FIGURES / 'rmse_by_model.png')
    plot_observed_vs_predicted(predictions, REGION_FIGURES / 'observed_vs_predicted.png')
    plot_improvement_by_basin(improvement, REGION_FIGURES / 'improvement_by_region.png')
    graph_path = REGION_OUTPUTS / 'edges_real_knn_undirected.csv'
    # Plot the learned spatial graph when edge outputs are present.
    if graph_path.exists():
        graph_edges = pd.read_csv(graph_path)
        basin_names = predictions[['basin_id', 'basin_name']].drop_duplicates()
        plot_graph_edges(graph_edges, basin_names, REGION_FIGURES / 'africa_graph.png')
    print(f'Saved figures to {REGION_FIGURES}')